# 10 – Performance data generator

Builds the large tables for performance tuning in the `workspace.perf` schema: bronze `trans` multiplied by 100 (105.6M rows), one table without skew and one with skew. The other layers are not changed.

## 1. Preparation: perf schema and trans profile

Creates the `workspace.perf` schema and checks the assumptions behind the multiplication.

How to read the output:

- `max_trans_id` is below 10,000,000; copied IDs are shifted with `trans_id + copy_no × 10,000,000`, so there are no collisions.

- `min_date` / `max_date` are within 1993–1998 (Berka format `YYMMDD`); each copy is shifted 6 years forward, so the periods do not overlap.

- The second table shows the 5 accounts with the most transactions; the hot account for skew is picked from them.

In [0]:
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.perf")

df_trans_bronze = spark.table("workspace.bronze.trans")

# Everything is a string in bronze; cast to int to get the numeric max
# Dates are YYMMDD strings, all in 19xx, so string min/max sort correctly
df_profile = df_trans_bronze.select(
    F.count("*").alias("rows"),
    F.max(F.col("trans_id").cast("int")).alias("max_trans_id"),
    F.min("date").alias("min_date"),
    F.max("date").alias("max_date"),
    F.countDistinct("account_id").alias("accounts"),
)
display(df_profile)

df_top_accounts = (
    df_trans_bronze.groupBy("account_id")
    .count()
    .orderBy(F.desc("count"))
    .limit(5)
)
display(df_top_accounts)

## 2. Generator and parameters

`src/perf_data_generator.py` builds the perf tables in three steps:

| Function | What it does |
|---|---|
| `to_perf_trans` | Casts bronze `trans` to typed columns (`trans_id`, `account_id` int; `trans_date` date; `amount`, `balance` decimal(12,2)) |
| `multiply_rows` | Copies the rows `factor` times; `trans_id + copy_no × 10,000,000`, dates shifted `copy_no × 6` years |
| `add_skew` | Moves ~30% of the copied rows (`copy_no > 0`) to the hot account; original rows are not changed |

`run` applies these steps and writes two tables: `trans_big` (without skew) and `trans_skewed` (with skew).

The parameters are in the cell below; the write is in section 3.

In [0]:
import os
import sys
import importlib

repo_root = os.path.dirname(os.getcwd())
src_path = os.path.join(repo_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

import silver_transforms
import perf_data_generator
importlib.reload(silver_transforms)
importlib.reload(perf_data_generator)

RUN_FACTOR = 100       # 1 original + 99 copies
HOT_ACCOUNT_ID = 8261  # account with the most transactions in bronze
SKEW_RATIO = 0.3       # share of copied rows moved to the hot account
SEED = 42

## 3. Write the tables (factor 100)

`run` writes the two tables under `workspace.perf` with `overwrite` (~1 min).

Why factor 100: with factor 5 (26.5 MB) there was no timing difference between the queries, and Q2 had no shuffle in EXPLAIN (see `11_perf_baseline`). The date range is 1993–2592 (the 6-year shift is kept; the data is synthetic, so future dates are accepted).

How to read the output: `rows` 105,632,000, `max_trans_id` 993,682,987, dates 1993-01-01 – 2592-12-31. `hot_rows`: 67,500 in `trans_big` (675 × 100), ~31.4M in `trans_skewed`. `numFiles` and `size_mb` show the physical layout of each table.

In [0]:
perf_data_generator.run(
    spark,
    factor=RUN_FACTOR,
    skew_ratio=SKEW_RATIO,
    hot_account_id=HOT_ACCOUNT_ID,
    seed=SEED,
)

df_details = None
for table_name in ["trans_big", "trans_skewed"]:
    full_name = f"{perf_data_generator.PERF_SCHEMA}.{table_name}"

    # Physical layout: file count and size
    df_detail = spark.sql(f"DESCRIBE DETAIL {full_name}").select(
        F.lit(table_name).alias("table_name"),
        "numFiles",
        F.round(F.col("sizeInBytes") / 1024 / 1024, 1).alias("size_mb"),
    )

    # Content check: rows, hot account, date and ID range
    df_stats = spark.table(full_name).agg(
        F.count("*").alias("rows"),
        F.count_if(F.col("account_id") == HOT_ACCOUNT_ID).alias("hot_rows"),
        F.min("trans_date").alias("min_date"),
        F.max("trans_date").alias("max_date"),
        F.max("trans_id").alias("max_trans_id"),
    )

    # Both sides have one row; crossJoin puts them side by side
    df_detail = df_detail.crossJoin(df_stats)

    if df_details is None:
        df_details = df_detail
    else:
        df_details = df_details.unionByName(df_detail)

display(df_details)